# 26 · Gaussian Mixture Models & the EM Algorithm
> **Part:** Part 4 · Unsupervised learning  
> **Family:** `Unsupervised · Clustering / density estimation (probabilistic)`

## Definition
A Gaussian Mixture Model says the data was generated by choosing one of K hidden Gaussian components (with probability π_k) and drawing a point from it.
Unlike k-means, clusters can be elliptical, differently sized, and assignments are **soft**: each point has a probability of belonging to every component.
It is fitted by **Expectation-Maximization (EM)**.

## Intuition
Chicken-and-egg: if you knew which component produced each point, estimating the Gaussians would be easy; if you knew the Gaussians, assigning points would be easy.
EM alternates: guess soft assignments with the current Gaussians (E-step), then re-fit the Gaussians using those soft assignments (M-step). The likelihood never decreases.


## The Math

```text
p(x) = Σ_k π_k N(x | μ_k, Σ_k)
    E-step:  r_ik = π_k N(xᵢ|μ_k,Σ_k) / Σ_j π_j N(xᵢ|μ_j,Σ_j)           (responsibility of component k for point i)
    M-step:  N_k = Σᵢ r_ik ;  π_k = N_k/n ;  μ_k = Σ r_ik xᵢ / N_k ;  Σ_k = Σ r_ik (xᵢ−μ_k)(xᵢ−μ_k)ᵀ / N_k
    log-likelihood  ℓ = Σᵢ log Σ_k π_k N(xᵢ|…): rises monotonically; EM finds a local optimum.
    Choosing K: BIC = −2ℓ + p·ln n (printed below). k-means is the special case of equal spherical covariances and hard assignments.
```


In [ ]:
# Setup & Imports
import os
import pathlib
import sys
from IPython.display import Image, display

# Ensure parent folder is in path to import mlatlas
parent_dir = pathlib.Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import numpy as np
import matplotlib.pyplot as plt
from mlatlas import data, viz, cli


In [ ]:
# Model Implementation / Helper Classes
def logpdf(X, mu, S):
    d = X - mu; return -.5 * (np.einsum("ij,jk,ik->i", d, np.linalg.inv(S), d) + np.linalg.slogdet(S)[1] + X.shape[1] * np.log(2 * np.pi))
def em(X, K, iters, seed=0):
    rng = np.random.default_rng(seed); n, d = X.shape; mu = X[rng.choice(n, K, replace=False)]; S = np.array([np.eye(d) * 2.] * K); pi = np.ones(K) / K; H = []
    for _ in range(iters):
        L = np.stack([np.log(pi[k]) + logpdf(X, mu[k], S[k]) for k in range(K)], 1); m = L.max(1, keepdims=True); ll = (m[:, 0] + np.log(np.exp(L - m).sum(1))).sum(); R = np.exp(L - m); R /= R.sum(1, keepdims=True)
        H.append((mu.copy(), S.copy(), R.copy(), ll)); Nk = R.sum(0); pi = Nk / n; mu = (R.T @ X) / Nk[:, None]
        S = np.array([((R[:, k, None] * (X - mu[k])).T @ (X - mu[k])) / Nk[k] + 1e-4 * np.eye(d) for k in range(K)])
    return H


In [ ]:
# Demonstration & Training Pipeline
def main(gif, quick):
    rng = np.random.default_rng(1); X = np.r_[rng.multivariate_normal([0, 0], [[1.5, 1.1], [1.1, 1.0]], 180), rng.multivariate_normal([4, 3], [[.8, -.5], [-.5, .9]], 150), rng.multivariate_normal([-1, 4], [[.4, 0], [0, 1.2]], 120)]
    bic = []
    for K in range(1, 7):
        h = em(X, K, 40, seed=2)[-1]; p = K * (2 + 3) + K - 1; bic.append(-2 * h[3] + p * np.log(len(X)))
    print("  BIC for K=1..6:", [int(b) for b in bic], "-> best K =", int(np.argmin(bic)) + 1)
    K = 3; H = em(X, K, 12 if quick else 40, seed=0); cols = np.array([to_rgb(c) for c in viz.PALETTE[:K]]); fig, (a1, a2) = viz.subplots(1, 2, w=11.5, h=4.6)
    ll = [h[3] for h in H]; fr = list(range(len(H))) if not quick else list(range(0, len(H), 2))

    def update(i):
        a1.clear(); a2.clear(); mu, S, R, _ = H[i]; a1.scatter(*X.T, s=16, c=R @ cols, edgecolor="white", linewidth=.3, zorder=3)
        for k in range(K): viz.ellipse(a1, mu[k], S[k], viz.PALETTE[k], 1, lw=2); viz.ellipse(a1, mu[k], S[k], viz.PALETTE[k], 2, lw=1, ls="--"); a1.scatter(*mu[k], marker="X", s=150, color=viz.PALETTE[k], edgecolor="k", zorder=6)
        a1.set(title=f"EM iteration {i + 1}: soft assignments (blended colours = uncertain)", xlim=(-5, 8), ylim=(-4, 8))
        a2.plot(range(1, i + 2), ll[:i + 1], "o-", ms=3, color=viz.PALETTE[2]); a2.set(xlim=(0, len(H) + 1), ylim=(min(ll) - 50, max(ll) + 50), title="Log-likelihood (never decreases)", xlabel="iteration")
    viz.animate(fig, update, fr, gif, fps=5)


In [ ]:
# Run the model demonstration (quick=True for fast interactive execution)
gif_path = f"../outputs/26_gaussian_mixture_em.gif"
main(gif=gif_path, quick=True)


## Visualizing the Learning Process

Three overlapping, differently-shaped clusters. Each frame is one EM iteration: ellipses (1σ and 2σ) move/rotate/stretch to fit; point colours blend by responsibility (mixed colours = uncertain).
Right: the log-likelihood increasing. The console prints BIC for K = 1…6.


In [ ]:
# Display the learning animation inline
media_file = f"../outputs/26_gaussian_mixture_em.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/26_gaussian_mixture_em.gif"):
    display(Image(filename=f"outputs/26_gaussian_mixture_em.gif"))


## Practical tips

- Initialise with k-means; regularise covariances (add εI) to avoid collapse onto a single point; restart several times.
- Used for soft clustering, density estimation, anomaly detection (low p(x)), speaker/background modelling.


## 🧪 Try This & Experiments

- Fit K=2 and K=5 and compare the BIC.
- Initialise two means at the same point: what happens?
